# Swarm-Augmented Time Series Forecasting (SATF) for Commercial Real Estate
## Combining MiroFish-Style Swarm Intelligence with Google TimesFM — a CCIM Workflow

**A novel approach that bridges qualitative market intelligence with quantitative foundation-model forecasting, applied to commercial real estate.**

### The Core Idea

CRE forecasting suffers from a familiar split. Quantitative models read the rent/vacancy/cap-rate history but stay blind to news. Experienced brokers, appraisers, and lenders read the market but can't put rigorous probability bands on it. This notebook bridges the two:

1. **MiroFish-style swarm** — A panel of CCIM-style CRE experts (broker, appraiser, lender, developer, REIT analyst, economist...) debate the market over several rounds, producing an evolving sentiment trajectory.
2. **Google TimesFM** — A decoder-only foundation model pre-trained on 400B+ real-world time points produces a zero-shot quantitative forecast of the rent series.
3. **Claude meta-synthesis** — Reconciles the numbers and the narrative into a unified prediction with an explanation a client (or investment committee) can act on.

We forecast **metro industrial submarket asking rents ($/SF NNN)** as the primary series, while vacancy, cap rates, net absorption, and financing conditions feed the scenario so the swarm reasons across the whole market. Swap in any asset class (office, retail, multifamily) by editing the scenario and data.

### What You'll Learn

- How to simulate a CCIM-style multi-expert market debate using the Claude API
- How to extract structured numerical sentiment trajectories from that debate
- How to combine qualitative market signals with a TimesFM quantitative rent forecast
- How to use Claude as a meta-reasoning layer to synthesize both into a client-ready call
- How dynamic shock injection (a Fed move, a major tenant event) reveals the limits of pure quantitative forecasting

### Table of Contents

1. [Setup and Installation](#setup)
2. [Define the Prediction Scenario](#scenario)
3. [Swarm Simulation — Multi-Expert Debate via Claude](#swarm)
4. [Sentiment Trajectory Extraction](#sentiment)
5. [Baseline TimesFM Forecast](#baseline)
6. [Swarm-Augmented Forecast](#augmented)
7. [Claude Meta-Synthesis — Unified Prediction](#synthesis)
8. [Visualization and Analysis](#visualization)

<a id="setup"></a>
## 1. Setup and Installation

We need:
- **`anthropic`** — Claude API for swarm simulation and meta-synthesis
- **`timesfm`** — Google's time series foundation model
- **`numpy`** / **`matplotlib`** — Numerical computation and visualization

In [ ]:
%pip install anthropic timesfm[torch] matplotlib numpy --quiet

In [ ]:
import json
import re

import anthropic
import matplotlib.pyplot as plt
import numpy as np

# Requires ANTHROPIC_API_KEY environment variable to be set
# See: https://docs.anthropic.com/en/docs/initial-setup
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

<a id="scenario"></a>
## 2. Define the Prediction Scenario

We'll forecast **metro industrial submarket asking rents ($/SF/yr, NNN)** over the next 12 months. Industrial is a good test case because:
- It has strong **quantitative signals** (a clean monthly rent history, vacancy, net absorption)
- And **qualitative drivers** that numbers miss (Fed rate moves, spec-construction pipeline, reshoring/e-commerce demand, large tenant moves, cap-rate repricing)

We'll use synthetic but realistic history. To use your own market, replace `historical_data` with your monthly rent series (CoStar / CompStak / internal comps) and update the `context` to your submarket. The same workflow applies to office, retail, or multifamily — change the asset class, the units, and the expert panel.

In [ ]:
# Scenario definition — this "reality seed" drives the swarm simulation
SCENARIO = {
    "topic": "Metro Industrial Submarket Asking Rents ($/SF/yr, NNN)",
    "context": """
    Current state (April 2026) — a primary-market industrial/logistics submarket:
    - Average asking rent is ~$13.00/SF NNN, up from ~$9.00 three years ago (cycle of rapid rent growth)
    - Direct vacancy has risen to ~5.2% from a cycle low of ~3.8% as new speculative supply delivers
    - Net absorption remains positive but is decelerating quarter over quarter
    - The construction pipeline is large: several million SF of spec product delivering over the next 12 months
    - Market cap rates have expanded ~125 bps from cycle lows to ~6.75% as financing costs rose
    - The 10-year Treasury remains elevated, pressuring valuations and slowing transaction volume
    - Demand tailwinds persist from e-commerce and reshoring/nearshoring of manufacturing
    - Concessions (free rent, TI allowances) are creeping up as landlords compete for credit tenants
    - Some occupiers are giving back space; tenant mix is shifting toward 3PL and advanced manufacturing
    """,
    "prediction_question": "What will average asking rent ($/SF/yr NNN) be over the next 12 months?",
    "forecast_horizon": 12,  # months
}

# Historical monthly asking rent ($/SF/yr NNN) — synthetic but realistic, 36 months of history
# Represents rapid rent growth maturing into a plateau, with mild seasonality and noise
np.random.seed(42)
months = np.arange(36)
trend = 9.0 + 4.0 * (1 / (1 + np.exp(-0.15 * (months - 18))))  # S-curve from ~$9 to ~$13
seasonal = 0.15 * np.sin(2 * np.pi * months / 12)  # Mild leasing-season pattern
noise = np.random.normal(0, 0.10, len(months))
historical_data = trend + seasonal + noise

print(f"Historical data: {len(historical_data)} months")
print(f"Latest value: ${historical_data[-1]:.2f}/SF NNN")
print(f"Range: ${historical_data.min():.2f} — ${historical_data.max():.2f}/SF NNN")

# Plot historical data
plt.figure(figsize=(10, 4))
plt.plot(months, historical_data, "b-o", markersize=3, label="Historical Asking Rent ($/SF NNN)")
plt.xlabel("Month")
plt.ylabel("Asking Rent ($/SF/yr NNN)")
plt.title("Metro Industrial Submarket — Historical Asking Rents")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="swarm"></a>
## 3. Swarm Simulation — Multi-Agent Debate via Claude

This is the MiroFish-inspired core. Instead of deploying the full MiroFish infrastructure (Node.js frontend, OASIS engine, Zep Cloud memory), we use **Claude to simulate the swarm dynamics directly**.

### How it works:
1. **Define diverse agent personas** — each with a unique background, bias, and reasoning style
2. **Run multi-round debates** — agents react to each other's arguments, shifting positions
3. **Extract structured scores** — each agent provides a numerical sentiment/confidence score per round
4. **Emergent behavior** — collective opinion evolves through social influence, not just averaging

This mirrors MiroFish's GraphRAG → Agent Generation → OASIS Simulation pipeline, but uses Claude as the simulation substrate.

In [ ]:
# Define the swarm: diverse CRE expert personas (CCIM-style market panel)
SWARM_AGENTS = [
    {
        "name": "Linda Marsh, CCIM",
        "role": "Investment Sales Broker",
        "bias": "data-driven moderate",
        "background": "25 years brokering industrial investment sales. Lives in the comps; "
        "thinks in cap rates, price/SF, and what buyers will actually underwrite today.",
    },
    {
        "name": "Frank DeLuca, MAI",
        "role": "Commercial Appraiser",
        "bias": "conservative realist",
        "background": "MAI appraiser who values industrial assets for lenders. Anchors to "
        "verifiable comps and is skeptical of momentum that isn't supported by closed deals.",
    },
    {
        "name": "Priya Nair",
        "role": "CRE Debt / Life-Co Lender",
        "bias": "risk-aware credit lens",
        "background": "Originates industrial mortgages for a life insurer. Focused on DSCR, "
        "rates, and refinance risk; rent growth only matters if it survives a stress test.",
    },
    {
        "name": "Marcus Rodriguez",
        "role": "Industrial Developer",
        "bias": "supply-side optimist",
        "background": "Develops spec logistics product. Knows the construction pipeline cold "
        "and how much new supply is about to compete for the same tenants.",
    },
    {
        "name": "Aisha Williams",
        "role": "REIT Acquisitions Analyst",
        "bias": "long-term structural bull",
        "background": "Underwrites acquisitions for a public industrial REIT. Views the market "
        "through capital flows, replacement cost, and durable demand from e-commerce.",
    },
    {
        "name": "Tom Brewer",
        "role": "Local Leasing / Tenant-Rep Broker",
        "bias": "ground-level pragmatist",
        "background": "Boots-on-the-ground leasing broker. Sees real-time tenant demand, "
        "rising concessions, and which deals are actually getting signed vs. asking rents.",
    },
    {
        "name": "Dr. Sarah Chen",
        "role": "CRE Market Economist",
        "bias": "macro-anchored moderate",
        "background": "Heads research at a brokerage. Connects rents to the 10-year Treasury, "
        "employment, absorption, and the construction pipeline rather than any single comp.",
    },
    {
        "name": "David Kowalski",
        "role": "Private Capital Investor / Syndicator",
        "bias": "contrarian value-seeker",
        "background": "Raises private capital for value-add industrial deals. Hunts for "
        "dislocation and is quick to flag when consensus optimism has gotten ahead of fundamentals.",
    },
]

print(f"Swarm size: {len(SWARM_AGENTS)} agents")
for agent in SWARM_AGENTS:
    print(f"  • {agent['name']} — {agent['role']} ({agent['bias']})")

### Run the Multi-Round Swarm Debate

Each round, all agents respond to the evolving debate. Like MiroFish's OASIS simulation, agents can shift positions based on arguments from others — creating emergent collective intelligence rather than static polling.

> **Cost note:** The swarm simulation makes ~40 API calls (8 agents x 5 rounds) plus 1 synthesis call. With `claude-sonnet-4-6`, this typically costs under $0.50 total. The shock injection section adds another ~40 calls. Budget roughly $1.00 for a full notebook run.

In [ ]:
def run_swarm_round(
    agents: list[dict[str, str]],
    scenario: dict,
    round_num: int,
    previous_debate: list[dict],
) -> list[dict]:
    """Run one round of the swarm debate, collecting each agent's response."""
    round_results = []

    # Build the debate context from previous rounds
    debate_history = ""
    if previous_debate:
        debate_history = "\n\n--- PREVIOUS ROUND ARGUMENTS ---\n"
        for entry in previous_debate[-len(agents) :]:  # Show last round's arguments
            debate_history += f"\n**{entry['name']}** ({entry['role']}): {entry['argument']}\n"
            debate_history += f"  Sentiment: {entry['sentiment']}/10 | "
            debate_history += f"Predicted change: {entry['predicted_direction']}\n"

    for agent in agents:
        prompt = f"""You are {agent["name"]}, a {agent["role"]}.

Background: {agent["background"]}
Your natural analytical bias: {agent["bias"]}

SCENARIO:
Topic: {scenario["topic"]}
{scenario["context"]}

Question: {scenario["prediction_question"]}

This is Round {round_num} of a multi-expert debate.
{debate_history}

Based on your expertise and the arguments you've heard from others, provide:

1. Your argument (2-3 sentences, be specific and cite reasoning)
2. Your SENTIMENT score (1-10 scale):
   - 1-3: Very bearish (expect rents to decline or roll over)
   - 4-5: Moderately bearish/cautious (flat to softening)
   - 6-7: Moderately bullish/optimistic (continued growth)
   - 8-10: Very bullish (expect strong rent acceleration)
3. Your predicted direction for asking rent ($/SF NNN) over 12 months: UP, FLAT, or DOWN
4. Your confidence in your prediction (0-100%)

You MUST respond in this exact JSON format:
{{
    "argument": "your argument here",
    "sentiment": <number 1-10>,
    "predicted_direction": "UP" or "FLAT" or "DOWN",
    "confidence": <number 0-100>
}}

If other agents made compelling points, you may shift your position. Be authentic to your persona."""

        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            temperature=0.8,  # Higher temperature for diverse agent personalities
            messages=[{"role": "user", "content": prompt}],
        )

        response_text = response.content[0].text.strip()

        # Parse JSON from the response — handle markdown code blocks
        json_match = re.search(r"\{[^{}]*\}", response_text, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group())
                parsed["name"] = agent["name"]
                parsed["role"] = agent["role"]
                parsed["round"] = round_num
                round_results.append(parsed)
            except json.JSONDecodeError:
                print(f"  Warning: Failed to parse response from {agent['name']}, skipping")
        else:
            print(f"  Warning: No JSON found in response from {agent['name']}, skipping")

    return round_results


# Run 5 rounds of debate (simulating MiroFish's temporal simulation steps)
NUM_ROUNDS = 5
all_debate_results = []

print(f"Running {NUM_ROUNDS}-round swarm debate with {len(SWARM_AGENTS)} agents...\n")

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"--- Round {round_num} ---")
    round_results = run_swarm_round(SWARM_AGENTS, SCENARIO, round_num, all_debate_results)
    all_debate_results.extend(round_results)

    # Show round summary
    sentiments = [r["sentiment"] for r in round_results]
    avg_sentiment = np.mean(sentiments)
    directions = [r["predicted_direction"] for r in round_results]
    print(f"  Avg sentiment: {avg_sentiment:.1f}/10")
    print(f"  Directions: {', '.join(directions)}")
    print()

<a id="sentiment"></a>
## 4. Sentiment Trajectory Extraction

Now we extract the key innovation: **converting qualitative swarm debate into a quantitative time series** that can augment TimesFM's forecast.

We compute per-round metrics:
- **Swarm sentiment** — Weighted average sentiment (confidence-weighted)
- **Consensus strength** — How much agents agree (inverse of variance)
- **Directional momentum** — Net bullish vs. bearish signal

In [ ]:
def extract_sentiment_trajectory(debate_results: list[dict], num_rounds: int) -> dict[str, list]:
    """Convert swarm debate results into quantitative sentiment trajectories."""
    trajectory = {
        "rounds": [],
        "avg_sentiment": [],
        "weighted_sentiment": [],
        "consensus_strength": [],
        "bullish_ratio": [],
        "avg_confidence": [],
        "sentiment_momentum": [],  # Change in sentiment between rounds
    }

    for r in range(1, num_rounds + 1):
        round_data = [d for d in debate_results if d["round"] == r]
        if not round_data:
            continue

        sentiments = [d["sentiment"] for d in round_data]
        confidences = [d["confidence"] / 100.0 for d in round_data]
        directions = [d["predicted_direction"] for d in round_data]

        # Confidence-weighted sentiment
        weights = np.array(confidences)
        weighted_avg = np.average(sentiments, weights=weights)

        # Consensus: inverse of coefficient of variation (higher = more agreement)
        sentiment_std = np.std(sentiments)
        consensus = 1.0 / (1.0 + sentiment_std)

        # Bullish ratio
        bullish = sum(1 for d in directions if d == "UP") / len(directions)

        trajectory["rounds"].append(r)
        trajectory["avg_sentiment"].append(np.mean(sentiments))
        trajectory["weighted_sentiment"].append(weighted_avg)
        trajectory["consensus_strength"].append(consensus)
        trajectory["bullish_ratio"].append(bullish)
        trajectory["avg_confidence"].append(np.mean(confidences))

        # Momentum (change from previous round)
        if len(trajectory["avg_sentiment"]) > 1:
            momentum = trajectory["avg_sentiment"][-1] - trajectory["avg_sentiment"][-2]
        else:
            momentum = 0.0
        trajectory["sentiment_momentum"].append(momentum)

    return trajectory


sentiment_trajectory = extract_sentiment_trajectory(all_debate_results, NUM_ROUNDS)

# Display the trajectory
print("Swarm Sentiment Trajectory:")
print(
    f"{'Round':<8} {'Sentiment':<12} {'Weighted':<12} {'Consensus':<12} {'Bullish%':<12} {'Momentum':<12}"
)
print("-" * 68)
for i, r in enumerate(sentiment_trajectory["rounds"]):
    print(
        f"{r:<8} "
        f"{sentiment_trajectory['avg_sentiment'][i]:<12.2f}"
        f"{sentiment_trajectory['weighted_sentiment'][i]:<12.2f}"
        f"{sentiment_trajectory['consensus_strength'][i]:<12.2f}"
        f"{sentiment_trajectory['bullish_ratio'][i]:<12.1%}"
        f"{sentiment_trajectory['sentiment_momentum'][i]:<12.2f}"
    )

In [ ]:
# Visualize the swarm sentiment evolution across debate rounds
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Sentiment over rounds
axes[0, 0].plot(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["avg_sentiment"],
    "ro-",
    label="Avg Sentiment",
)
axes[0, 0].plot(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["weighted_sentiment"],
    "bs-",
    label="Confidence-Weighted",
)
axes[0, 0].axhline(y=5.5, color="gray", linestyle="--", alpha=0.5, label="Neutral")
axes[0, 0].set_xlabel("Debate Round")
axes[0, 0].set_ylabel("Sentiment (1-10)")
axes[0, 0].set_title("Swarm Sentiment Evolution")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Consensus strength
axes[0, 1].bar(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["consensus_strength"],
    color="green",
    alpha=0.7,
)
axes[0, 1].set_xlabel("Debate Round")
axes[0, 1].set_ylabel("Consensus Strength")
axes[0, 1].set_title("Agent Consensus (Higher = More Agreement)")
axes[0, 1].grid(True, alpha=0.3)

# Bullish ratio
axes[1, 0].bar(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["bullish_ratio"],
    color="blue",
    alpha=0.7,
)
axes[1, 0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
axes[1, 0].set_xlabel("Debate Round")
axes[1, 0].set_ylabel("Bullish Ratio")
axes[1, 0].set_title("% of Agents Predicting UP")
axes[1, 0].set_ylim(0, 1)
axes[1, 0].grid(True, alpha=0.3)

# Individual agent sentiment trajectories (the emergent behavior view)
for agent in SWARM_AGENTS:
    agent_data = [d for d in all_debate_results if d["name"] == agent["name"]]
    rounds = [d["round"] for d in agent_data]
    sents = [d["sentiment"] for d in agent_data]
    axes[1, 1].plot(rounds, sents, "o-", label=agent["name"], alpha=0.7, markersize=4)
axes[1, 1].set_xlabel("Debate Round")
axes[1, 1].set_ylabel("Sentiment (1-10)")
axes[1, 1].set_title("Individual Agent Trajectories")
axes[1, 1].legend(fontsize=6, loc="best")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<a id="baseline"></a>
## 5. Baseline TimesFM Forecast

Now we run Google TimesFM 2.5 on the raw historical data to get a **pure quantitative baseline** forecast. This is what you'd get without any swarm intelligence — just the numbers.

In [ ]:
import timesfm

try:
    import torch

    torch.set_float32_matmul_precision("high")
except ImportError:
    torch = None

# TimesFM ships two API generations; auto-detect whichever pip installed so the
# notebook runs whether you get the 2.5 release or the older 1.x line.
if hasattr(timesfm, "TimesFM_2p5_200M_torch"):
    _TFM_API = "2.5"
    tfm_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
    tfm_model.compile(
        timesfm.ForecastConfig(
            max_context=512,
            max_horizon=SCENARIO["forecast_horizon"],
            normalize_inputs=True,
            use_continuous_quantile_head=True,  # Enable probabilistic forecasts
            force_flip_invariance=True,
            infer_is_positive=True,
            fix_quantile_crossing=True,
        )
    )
elif hasattr(timesfm, "TimesFm"):
    _TFM_API = "1.x"
    tfm_model = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(
            backend="cpu",
            per_core_batch_size=32,
            horizon_len=SCENARIO["forecast_horizon"],
            context_len=512,
        ),
        checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id="google/timesfm-1.0-200m-pytorch"),
    )
else:
    raise ImportError("Upgrade timesfm: pip install --upgrade 'timesfm[torch]'")


def tfm_forecast(history, horizon):
    """Forecast with either TimesFM API generation.

    Returns (point_forecast[horizon], quantile_forecast[horizon, n_quantiles]).
    """
    series = np.asarray(history, dtype=float)
    if _TFM_API == "2.5":
        point, quantiles = tfm_model.forecast(horizon=horizon, inputs=[series])
        return np.asarray(point[0])[:horizon], np.asarray(quantiles[0])[:horizon]
    # 1.x: column 0 of the quantile output is the mean; drop it so column 0 is a low quantile.
    point, quantiles = tfm_model.forecast([series], freq=[0])
    return np.asarray(point[0])[:horizon], np.asarray(quantiles[0])[:horizon, 1:]


print(f"TimesFM loaded successfully (API {_TFM_API})")

In [ ]:
# Run baseline forecast on historical data (works with either TimesFM API)
baseline_forecast, baseline_quantile_forecast = tfm_forecast(
    historical_data, SCENARIO["forecast_horizon"]
)

forecast_months = np.arange(
    len(historical_data), len(historical_data) + SCENARIO["forecast_horizon"]
)

print(f"Baseline forecast (next {SCENARIO['forecast_horizon']} months):")
for i, val in enumerate(baseline_forecast):
    print(f"  Month {i + 1}: ${val:.2f}/SF NNN")

# Plot baseline forecast
plt.figure(figsize=(12, 5))
plt.plot(months, historical_data, "b-o", markersize=3, label="Historical")
plt.plot(
    forecast_months, baseline_forecast, "r--o", markersize=3, label="TimesFM Baseline Forecast"
)

# Plot confidence bands from quantile forecasts
if baseline_quantile_forecast.ndim == 2 and baseline_quantile_forecast.shape[1] >= 2:
    lower = baseline_quantile_forecast[:, 0]  # Lower quantile
    upper = baseline_quantile_forecast[:, -1]  # Upper quantile
    plt.fill_between(
        forecast_months, lower, upper, alpha=0.2, color="red", label="Prediction Interval"
    )

plt.xlabel("Month")
plt.ylabel("Asking Rent ($/SF/yr NNN)")
plt.title("TimesFM Baseline Forecast — Pure Quantitative")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="augmented"></a>
## 6. Swarm-Augmented Forecast

**This is the novel combination.** We use the swarm's sentiment trajectory to modify TimesFM's baseline forecast:

### The SATF Algorithm:

1. **Sentiment Modifier** — The swarm's final consensus sentiment maps to a directional bias:
   - Sentiment > 5.5 → bullish modifier (scales forecast upward)
   - Sentiment < 5.5 → bearish modifier (scales forecast downward)
   - The magnitude depends on consensus strength (high agreement = stronger signal)

2. **Confidence Band Adjustment** — Low swarm consensus *widens* the prediction interval (more uncertainty), high consensus *tightens* it

3. **Momentum Overlay** — If swarm sentiment shifted strongly during debate, apply a trend acceleration/deceleration to the forecast curve

This bridges the qualitative-quantitative gap: **the swarm tells us WHERE the numbers might be wrong.**

In [ ]:
def swarm_augmented_forecast(
    baseline_forecast: np.ndarray,
    baseline_quantiles: np.ndarray,
    sentiment_trajectory: dict[str, list],
) -> tuple[np.ndarray, np.ndarray, dict]:
    """Apply swarm intelligence signals to modify TimesFM's quantitative forecast."""
    horizon = len(baseline_forecast)

    # Extract final-round swarm signals
    final_sentiment = sentiment_trajectory["weighted_sentiment"][-1]
    final_consensus = sentiment_trajectory["consensus_strength"][-1]
    final_bullish_ratio = sentiment_trajectory["bullish_ratio"][-1]

    # Compute sentiment momentum (trend across all rounds)
    sentiments = sentiment_trajectory["weighted_sentiment"]
    if len(sentiments) >= 2:
        sentiment_trend = (sentiments[-1] - sentiments[0]) / len(sentiments)
    else:
        sentiment_trend = 0.0

    # --- 1. Directional Bias ---
    # Map sentiment (1-10) to a multiplier centered on 1.0
    # Sentiment of 5.5 = neutral (1.0x), 10 = 1.05x, 1 = 0.95x
    sentiment_deviation = (final_sentiment - 5.5) / 4.5  # Range: [-1, 1]
    # Scale by consensus: high agreement amplifies the signal
    max_adjustment = 0.05  # Maximum 5% adjustment to forecast
    directional_bias = sentiment_deviation * final_consensus * max_adjustment

    # Apply bias with increasing strength over the forecast horizon
    # (swarm insights matter more for longer-term predictions)
    time_weights = np.linspace(0.3, 1.0, horizon)
    augmented_forecast = baseline_forecast * (1 + directional_bias * time_weights)

    # --- 2. Momentum Overlay ---
    # If the swarm's sentiment shifted during debate, apply acceleration
    momentum_factor = sentiment_trend * 0.002  # Small but compounding
    momentum_curve = np.cumsum(np.full(horizon, momentum_factor))
    augmented_forecast = augmented_forecast + momentum_curve

    # --- 3. Confidence Band Adjustment ---
    augmented_quantiles = baseline_quantiles.copy()
    if augmented_quantiles.ndim == 2 and augmented_quantiles.shape[1] >= 2:
        # Low consensus = wider bands, high consensus = tighter bands
        band_scale = 1.0 + (1.0 - final_consensus) * 0.5  # 1.0 to 1.5x width
        mid = augmented_forecast
        for q in range(augmented_quantiles.shape[1]):
            augmented_quantiles[:, q] = (
                mid + (augmented_quantiles[:, q] - baseline_forecast) * band_scale
            )

    return (
        augmented_forecast,
        augmented_quantiles,
        {
            "directional_bias": directional_bias,
            "sentiment_trend": sentiment_trend,
            "final_sentiment": final_sentiment,
            "final_consensus": final_consensus,
            "final_bullish_ratio": final_bullish_ratio,
        },
    )


augmented_forecast, augmented_quantiles, swarm_signals = swarm_augmented_forecast(
    baseline_forecast, baseline_quantile_forecast, sentiment_trajectory
)

print("Swarm Augmentation Signals:")
print(f"  Final sentiment: {swarm_signals['final_sentiment']:.2f}/10")
print(f"  Consensus strength: {swarm_signals['final_consensus']:.3f}")
print(f"  Bullish ratio: {swarm_signals['final_bullish_ratio']:.1%}")
print(f"  Directional bias: {swarm_signals['directional_bias']:+.4f}")
print(f"  Sentiment trend: {swarm_signals['sentiment_trend']:+.3f}/round")
print()
print("Forecast Comparison:")
print(f"{'Month':<8} {'Baseline':<12} {'Augmented':<12} {'Difference':<12}")
print("-" * 44)
for i in range(len(baseline_forecast)):
    diff = augmented_forecast[i] - baseline_forecast[i]
    print(f"{i + 1:<8} {baseline_forecast[i]:<12.2f} {augmented_forecast[i]:<12.2f} {diff:<+12.3f}")

<a id="shock"></a>
## 6b. Dynamic Shock Injection

**This is what makes SATF truly dynamic.** We inject a surprise event mid-forecast and re-run the swarm:

- **TimesFM's baseline doesn't change** — it only sees historical numbers, not news
- **The swarm adapts immediately** — agents reason about the shock and shift positions
- **The augmented forecast diverges** from baseline, capturing the shock's real-world impact

This is the scenario pure quantitative models can never handle: unprecedented events that break historical patterns.

In [ ]:
# Define a mid-forecast shock event
SHOCK_EVENT = """
BREAKING NEWS (Month 6 of forecast period):
The Federal Reserve delivers a surprise 75 bps rate cut and signals more easing,
sending the 10-year Treasury sharply lower and reopening the financing market.
Simultaneously, a major automaker announces a 5,000,000 SF build-to-suit advanced
manufacturing campus in this submarket, anchoring a reshoring supercluster and
pulling forward supplier demand. Two large speculative projects that were set to
deliver are paused, tightening the near-term supply pipeline.
"""

# Create a shocked scenario by appending the event
shocked_scenario = {
    "topic": SCENARIO["topic"],
    "context": SCENARIO["context"] + "\n\nSHOCK EVENT:\n" + SHOCK_EVENT,
    "prediction_question": (
        "Given this shock event, how will average asking rent ($/SF/yr NNN) "
        "change over the REMAINING 6 months of the forecast period?"
    ),
    "forecast_horizon": SCENARIO["forecast_horizon"],
}

# Re-run the swarm with the shock injected
print("Re-running swarm with SHOCK EVENT injected...\n")
print(SHOCK_EVENT)

shock_debate_results = []
for round_num in range(1, NUM_ROUNDS + 1):
    print(f"--- Shock Round {round_num} ---")
    round_results = run_swarm_round(SWARM_AGENTS, shocked_scenario, round_num, shock_debate_results)
    shock_debate_results.extend(round_results)

    sentiments = [r["sentiment"] for r in round_results]
    print(f"  Avg sentiment: {np.mean(sentiments):.1f}/10")
    print()

shock_trajectory = extract_sentiment_trajectory(shock_debate_results, NUM_ROUNDS)

In [ ]:
# Compare pre-shock vs post-shock swarm signals
shock_augmented, shock_quantiles, shock_signals = swarm_augmented_forecast(
    baseline_forecast, baseline_quantile_forecast, shock_trajectory
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Sentiment comparison
axes[0].plot(
    sentiment_trajectory["rounds"],
    sentiment_trajectory["weighted_sentiment"],
    "b-o",
    label="Original Swarm",
    linewidth=2,
)
axes[0].plot(
    shock_trajectory["rounds"],
    shock_trajectory["weighted_sentiment"],
    "r-s",
    label="Post-Shock Swarm",
    linewidth=2,
)
axes[0].axhline(y=5.5, color="gray", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Debate Round")
axes[0].set_ylabel("Weighted Sentiment (1-10)")
axes[0].set_title("Swarm Sentiment: Original vs Post-Shock")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Forecast comparison
axes[1].plot(
    forecast_months,
    baseline_forecast,
    "k--",
    label="TimesFM Baseline (unchanged)",
    linewidth=1.5,
)
axes[1].plot(
    forecast_months,
    augmented_forecast,
    "b-o",
    label="Original SATF",
    markersize=4,
)
axes[1].plot(
    forecast_months,
    shock_augmented,
    "r-s",
    label="Post-Shock SATF",
    markersize=4,
    linewidth=2,
)

# Shade the shock impact zone
shock_delta = shock_augmented - augmented_forecast
axes[1].fill_between(
    forecast_months,
    augmented_forecast,
    shock_augmented,
    alpha=0.2,
    color="red",
    label="Shock Impact Zone",
)
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Asking Rent ($/SF/yr NNN)")
axes[1].set_title("Forecast Impact of Shock Event")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle(
    "Dynamic Shock Injection: TimesFM stays blind, the Swarm adapts",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("\nShock Impact Summary:")
print(f"  Original final sentiment:   {swarm_signals['final_sentiment']:.2f}/10")
print(f"  Post-shock final sentiment: {shock_signals['final_sentiment']:.2f}/10")
print(f"  Forecast shift at Month 12: ${shock_delta[-1]:+.3f}/SF")

<a id="synthesis"></a>
## 7. Claude Meta-Synthesis — Unified Prediction

The final piece: Claude acts as the **meta-reasoning layer**, consuming both the quantitative forecast and the qualitative swarm debate to produce a unified narrative prediction.

This is something neither MiroFish nor TimesFM can do alone — bridging numbers and narrative into actionable insight.

In [ ]:
# Prepare the full context for Claude's meta-synthesis
# Collect final-round arguments from all agents
final_round_args = [d for d in all_debate_results if d["round"] == NUM_ROUNDS]
debate_summary = "\n".join(
    f"- {d['name']} ({d['role']}, sentiment {d['sentiment']}/10): {d['argument']}"
    for d in final_round_args
)

synthesis_prompt = f"""You are a meta-analyst synthesizing two independent prediction signals into a unified CRE forecast for an investment committee.

## SIGNAL 1: Quantitative Forecast (Google TimesFM 2.5)
A time series foundation model trained on 400B+ data points analyzed 36 months of historical asking rents for a metro industrial submarket ($/SF/yr NNN).

Baseline forecast (next 12 months, monthly asking rent $/SF NNN):
{chr(10).join(f"  Month {i + 1}: ${v:.2f}/SF" for i, v in enumerate(baseline_forecast))}

## SIGNAL 2: Swarm Intelligence (CCIM-Style Multi-Expert Debate)
8 CRE experts (broker, appraiser, lender, developer, REIT analyst, leasing broker, economist, private investor) debated for {NUM_ROUNDS} rounds.

Final sentiment trajectory:
- Starting swarm sentiment: {sentiment_trajectory["weighted_sentiment"][0]:.2f}/10
- Final swarm sentiment: {sentiment_trajectory["weighted_sentiment"][-1]:.2f}/10
- Consensus strength: {sentiment_trajectory["consensus_strength"][-1]:.3f}
- Bullish ratio: {sentiment_trajectory["bullish_ratio"][-1]:.1%}

Final round expert positions:
{debate_summary}

## SWARM-AUGMENTED FORECAST
After applying swarm signals to the baseline:
{chr(10).join(f"  Month {i + 1}: ${v:.2f}/SF (delta: {v - baseline_forecast[i]:+.3f})" for i, v in enumerate(augmented_forecast))}

## YOUR TASK
Produce a meta-synthesis for a CCIM-level audience that:
1. Identifies where the quantitative model and the expert swarm AGREE (high-confidence zones)
2. Identifies where they DIVERGE (areas of maximum uncertainty)
3. Explains which qualitative market factors the pure numbers miss (supply pipeline, rate moves, concessions, tenant credit, cap-rate repricing)
4. Provides your unified 12-month asking-rent prediction ($/SF NNN) with confidence levels
5. Flags the top 3 "wild card" events that could break the forecast, and notes the implication for valuation/underwriting

Be specific. Cite which experts' arguments were most compelling and why.
Reference specific months where the two signals agree or diverge."""

synthesis = client.messages.create(
    model=MODEL,
    max_tokens=2000,
    temperature=0.3,  # Lower temperature for analytical synthesis
    messages=[{"role": "user", "content": synthesis_prompt}],
)

print("=" * 80)
print("CLAUDE META-SYNTHESIS: Unified Prediction Report")
print("=" * 80)
print(synthesis.content[0].text)

<a id="visualization"></a>
## 8. Final Visualization — The Complete SATF Picture

Putting it all together: historical data, baseline TimesFM forecast, and the swarm-augmented forecast side by side.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})

# --- Top panel: Main forecast comparison ---
ax1 = axes[0]
ax1.plot(months, historical_data, "b-o", markersize=3, linewidth=1.5, label="Historical Data")
ax1.plot(
    forecast_months,
    baseline_forecast,
    "r--s",
    markersize=4,
    linewidth=1.5,
    label="TimesFM Baseline",
)
ax1.plot(
    forecast_months,
    augmented_forecast,
    "g-D",
    markersize=4,
    linewidth=2,
    label="Swarm-Augmented (SATF)",
)

# Baseline confidence band
if baseline_quantile_forecast.ndim == 2 and baseline_quantile_forecast.shape[1] >= 2:
    ax1.fill_between(
        forecast_months,
        baseline_quantile_forecast[:, 0],
        baseline_quantile_forecast[:, -1],
        alpha=0.1,
        color="red",
        label="Baseline CI",
    )

# Augmented confidence band
if augmented_quantiles.ndim == 2 and augmented_quantiles.shape[1] >= 2:
    ax1.fill_between(
        forecast_months,
        augmented_quantiles[:, 0],
        augmented_quantiles[:, -1],
        alpha=0.15,
        color="green",
        label="Swarm-Augmented CI",
    )

# Mark the transition point
ax1.axvline(x=len(historical_data) - 0.5, color="gray", linestyle=":", alpha=0.5)
ax1.text(
    len(historical_data) + 0.5,
    ax1.get_ylim()[1] * 0.95,
    "Forecast",
    fontsize=9,
    color="gray",
)

ax1.set_xlabel("Month")
ax1.set_ylabel("Asking Rent ($/SF/yr NNN)")
ax1.set_title(
    "Swarm-Augmented Time Series Forecasting (SATF) — CRE Industrial Rents\n"
    "CCIM-Style Expert Swarm + Google TimesFM 2.5 + Claude Meta-Synthesis",
    fontsize=13,
)
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# --- Bottom panel: Swarm signal overlay ---
ax2 = axes[1]

# Show sentiment trajectory alongside the forecast divergence
ax2_left = ax2
rounds_x = np.linspace(forecast_months[0], forecast_months[-1], len(sentiment_trajectory["rounds"]))
ax2_left.bar(
    rounds_x,
    sentiment_trajectory["weighted_sentiment"],
    width=0.6,
    color="purple",
    alpha=0.6,
    label="Swarm Sentiment",
)
ax2_left.axhline(y=5.5, color="gray", linestyle="--", alpha=0.5)
ax2_left.set_ylabel("Swarm Sentiment (1-10)", color="purple")
ax2_left.set_xlabel("Forecast Month")
ax2_left.set_ylim(0, 10)
ax2_left.tick_params(axis="y", labelcolor="purple")

# Overlay forecast divergence on right axis
ax2_right = ax2.twinx()
divergence = augmented_forecast - baseline_forecast
ax2_right.plot(
    forecast_months,
    divergence,
    "g-o",
    markersize=3,
    linewidth=1.5,
    label="Forecast Divergence",
)
ax2_right.axhline(y=0, color="gray", linestyle="-", alpha=0.3)
ax2_right.set_ylabel("SATF - Baseline ($/SF)", color="green")
ax2_right.tick_params(axis="y", labelcolor="green")

# Combined legend
lines1, labels1 = ax2_left.get_legend_handles_labels()
lines2, labels2 = ax2_right.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

### What You Learned

In this notebook you:

1. **Simulated a CCIM-style expert swarm** using Claude — 8 CRE experts debating across 5 rounds with emergent opinion shifts
2. **Extracted quantitative sentiment trajectories** from qualitative multi-expert debate (weighted sentiment, consensus strength, bullish ratio)
3. **Ran zero-shot time series forecasting** with Google TimesFM 2.5 on historical industrial asking rents
4. **Combined both signals** using the SATF algorithm — directional bias, momentum overlay, and confidence band adjustment
5. **Demonstrated dynamic shock injection** — showing how the expert swarm adapts to a Fed move + major tenant event while the pure quantitative model stays blind
6. **Used Claude as a meta-synthesis layer** to reconcile the quantitative forecast with the qualitative market narrative into a client-ready call

### Related Resources

- [MiroFish — Swarm Intelligence Engine](https://github.com/666ghj/MiroFish)
- [Google TimesFM — Time Series Foundation Model](https://github.com/google-research/timesfm)
- [Anthropic Claude API Documentation](https://docs.anthropic.com)

## What Makes This Novel

**Combining these three systems for CRE forecasting:**

| Component | Role | What It Provides |
|---|---|---|
| **CCIM-Style Expert Swarm** (via Claude) | Qualitative intelligence | Emergent sentiment trajectories from diverse CRE experts (broker, appraiser, lender, developer, REIT analyst...) debating over rounds |
| **Google TimesFM 2.5** | Quantitative forecasting | Zero-shot point + quantile rent forecasts from a foundation model trained on 400B+ data points |
| **Claude Meta-Synthesis** | Reasoning layer | Reconciliation of where the numbers and the market narrative agree/diverge, with explanation for an investment committee |

### The SATF Framework

**Swarm-Augmented Time Series Forecasting** bridges the qualitative-quantitative gap:

1. **Swarm signals modify forecast direction** — When experts collectively see a move the numbers miss (a Fed pivot, a wave of supply, a major tenant signing), the forecast shifts accordingly
2. **Consensus modulates confidence bands** — Disagreement among experts *widens* prediction intervals; agreement *narrows* them
3. **Meta-synthesis explains the forecast** — Instead of a black-box number, you get a narrative explaining which market factors drive the prediction

### Extensions for CRE

- **Real data feeds** — Replace synthetic history with your own monthly comps (CoStar / CompStak / internal lease data)
- **Multi-metric forecasting** — Run the same workflow on vacancy, net absorption, cap rates, or sale price/SF and cross-check the signals
- **Submarket / asset-class transfer** — Swap the scenario and expert panel for office, retail, multifamily, or a specific MSA
- **Scenario analysis** — Run multiple swarm simulations with different "reality seeds" (rates up vs. down) and compare forecast distributions
- **Underwriting integration** — Feed the unified rent path and confidence bands directly into a DCF / argus model for valuation